# ADP PM Notebook 4 — Agents, Tools, Memory and Skills

**Purpose:** Combine the original Strands agent fundamentals and skills notebooks into one short PM-friendly notebook.

**Retained:** agent loop mental model, brain-only agent, tool use, simple memory, one programmatic Skill, skill design checklist.



## PM mental model

An agent is not just a chatbot.

> Agent = LLM + instructions + tools/actions + memory + skills + guardrails.

This notebook prepares learners for the Copilot/no-code agent workflow in Day 2.


In [ ]:
# Uncomment if needed.
# %pip install -q strands-agents strands-agents-tools boto3 pandas pydantic==2.9


In [ ]:
import boto3
import pandas as pd
from datetime import datetime

from strands import Agent, tool
from strands.models import BedrockModel
from strands.vended_plugins.skills.agent_skills import AgentSkills
from strands.vended_plugins.skills.skill import Skill

AWS_REGION = boto3.session.Session().region_name or "us-east-1"
MODEL_ID = "amazon.nova-lite-v1:0"

bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.2,
)

print("Region:", AWS_REGION)
print("Model:", MODEL_ID)


In [ ]:
def show_response(title, response):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    print(response)


## 1. Agent with brain only

The agent can reason and respond, but cannot take action outside the model.


In [ ]:
concept_agent = Agent(
    model=bedrock_model,
    system_prompt="""
You are a product-manager friendly AI teacher.
Explain concepts simply and avoid deep implementation detail.
""",
)

response = concept_agent("Explain the difference between an LLM and an AI agent in five bullets.")
show_response("Brain-only agent", response)


## 2. Agent with tools

Tools let the agent call approved functions. In products, this is where permissions, approvals, and logging become important.


In [ ]:
@tool
def get_current_class_time() -> str:
    """Return the current date and time for the classroom session."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@tool
def calculate_percentage(score: float, total: float) -> str:
    """Calculate percentage score from score and total marks."""
    if total == 0:
        return "Total cannot be zero."
    return f"{(score / total) * 100:.2f}%"


tool_agent = Agent(
    model=bedrock_model,
    tools=[get_current_class_time, calculate_percentage],
    system_prompt="""
You are a classroom assistant.
Use tools when calculation or current class time is requested.
Explain which tool you used in simple language.
""",
)

response = tool_agent("A learner scored 42 out of 50. Calculate the percentage and tell the current class time.")
show_response("Agent with tools", response)


## 3. Agent with simple memory

This demonstrates the idea of memory. In enterprise products, memory requires consent, retention rules, and auditability.


In [ ]:
agent_memory = {}

@tool
def remember_preference(name: str, preference: str) -> str:
    """Save a user's preference in memory."""
    agent_memory[name.lower()] = preference
    return f"Saved preference for {name}: {preference}"


@tool
def recall_preference(name: str) -> str:
    """Recall a user's saved preference."""
    preference = agent_memory.get(name.lower())
    if preference is None:
        return f"No preference found for {name}."
    return f"{name}'s remembered preference is: {preference}"

memory_agent = Agent(
    model=bedrock_model,
    tools=[remember_preference, recall_preference],
    system_prompt="""
You are a safe memory demo agent.
Use memory tools only when the user explicitly asks you to remember or recall a preference.
Do not invent memory.
""",
)

show_response("Turn 1", memory_agent("Remember that Meera prefers concise PRD feedback."))
show_response("Turn 2", memory_agent("What does Meera prefer?"))
print("Memory store:", agent_memory)


## 4. Agent with one Skill

A Skill packages repeatable expertise. For PMs, this maps well to Copilot agent instructions or reusable prompt capabilities.


In [ ]:
policy_simplifier_skill = Skill(
    name="policy-simplifier",
    description="Rewrite HR or admin policy text into simple employee-friendly language.",
    instructions="""
You are the Policy Simplifier Skill.
When used:
1. Rewrite complex policy text in simple language.
2. Keep the original meaning.
3. Use this format:
   - Simple version
   - Key rules
   - Employee action needed
""",
)

policy_skill_plugin = AgentSkills(skills=[policy_simplifier_skill])

policy_agent = Agent(
    model=bedrock_model,
    plugins=[policy_skill_plugin],
    system_prompt="Use the available skill when the user asks to simplify HR, admin, or policy text.",
)

policy_text = """
Employees may work remotely up to two days per week subject to manager approval,
business continuity requirements, data security obligations, and client confidentiality constraints.
"""

response = policy_agent(f"Please simplify this policy for employees:\n\n{policy_text}")
show_response("Policy Skill output", response)
print("Activated skills:", policy_skill_plugin.get_activated_skills(policy_agent))


## 5. Skill and agent design checklist

Use this before moving into VS Code Copilot or Copilot Studio/no-code workflows.


In [ ]:
checklist = pd.DataFrame([
    {"Design area": "Purpose", "PM question": "What business job should the agent do?"},
    {"Design area": "Instructions", "PM question": "What role, tone, rules, and boundaries should it follow?"},
    {"Design area": "Tools/actions", "PM question": "What systems can it call, and what requires approval?"},
    {"Design area": "Knowledge", "PM question": "Which documents or sources are trusted?"},
    {"Design area": "Memory", "PM question": "What may be remembered, for how long, and with what consent?"},
    {"Design area": "Fallback", "PM question": "What should it do when uncertain or unauthorized?"},
    {"Design area": "Evaluation", "PM question": "How do we test correctness, safety, latency, and cost?"},
])
display(checklist)
